# How is the scaffold/harness making the model perform better or worse? 

In [ ]:
import json
import os
from pathlib import Path
import re, math, copy, warnings, hashlib, time, textwrap
from collections import Counter, defaultdict
from dotenv import load_dotenv

import pandas as pd
import numpy as np
from scipy.stats import binom, binomtest
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
#from adjustText import adjust_text
import seaborn as sns

SCAFFOLD_COLORS = {
    "Codex CLI":    "#0173b2",
    "Claude Code":  "#de8f05",
    "OpenCode":     "#029e73",
    "CORE-Agent":   "#cc78bc",
    "OpenCode":     "#029e73",
}

In [ ]:
DATA_DIR = Path("run_data")
FOCUS_CACHE = DATA_DIR / "raw_focus.json"
assert FOCUS_CACHE.exists(), f"Cache not found: {FOCUS_CACHE}. Run the cache cell in model_scaffold_decomposition.ipynb first."

with open(FOCUS_CACHE) as f:
    rows = json.load(f)
print(f"Loaded {len(rows)} runs from {FOCUS_CACHE}")

In [ ]:
# ── Build run-level DataFrame ─────────────────────────────────────────────────
# Single pass over all 390 runs: extract metadata + transcript-derived features.

MODEL_MAP = {
    "claude-opus-4-5": "Opus 4.5",
    "claude-opus-4-6": "Opus 4.6",
    "gpt-5.4": "GPT-5.4",
}

MODEL_COLORS = {"Opus 4.5": "#5B8DEF", "Opus 4.6": "#1A3A8F", "GPT-5.4": "#FF6B35"}
HARNESS_MARKERS = {"Claude Code": "o", "CORE-Agent": "s", "Codex CLI": "D", "OpenCode": "^"}
PASS_FAIL_COLORS = {True: "#4CAF50", False: "#E53935"}

sns.set_style("whitegrid")
plt.rcParams.update({"figure.dpi": 150, "savefig.dpi": 300, "font.size": 11})
os.makedirs("figures", exist_ok=True)

records = []
for r in rows:
    meta = r["agent_run_metadata"]
    sm = meta.get("scoring_metadata", {})
    if isinstance(sm, str):
        sm = json.loads(sm)

    model_raw = meta.get("model", "")
    model = MODEL_MAP.get(model_raw, model_raw)
    harness = meta.get("harness", "")
    effort = meta.get("reasoning_effort") or "default"
    capsule_id = meta.get("capsule_id", "").replace("capsule-", "")

    if harness == "Codex CLI":
        config = f"{model} / {harness} ({effort})"
    elif effort != "default" and harness == "Claude Code":
        config = f"{model} / {harness}"
    else:
        config = f"{model} / {harness}"

    # Transcript-derived features (single pass)
    assistant_msg_count = 0
    total_assistant_chars = 0
    context_resets = 0
    tool_counts = Counter()

    for m in r.get("messages", []):
        if m.get("role") != "assistant":
            continue
        content = m.get("content", "")
        if isinstance(content, list):
            content = " ".join(c.get("text", "") for c in content if isinstance(c, dict))

        assistant_msg_count += 1
        total_assistant_chars += len(content)

        if "I still need to solve the task" in content:
            context_resets += 1

        tool_counts["execute_bash"] += len(re.findall(r"execute_bash\(", content))
        tool_counts["inspect_file"] += len(re.findall(r"inspect_file_as_text\(", content))
        tool_counts["vision_query"] += len(re.findall(r"query_vision_language_model\(", content))
        tool_counts["edit_file"] += len(re.findall(r"edit_file\(", content))
        tool_counts["final_answer"] += len(re.findall(r"final_answer\(", content))

    records.append({
        "model": model,
        "harness": harness,
        "reasoning_effort": effort,
        "config": config,
        "capsule_id": capsule_id,
        "message_count": meta.get("message_count", 0),
        "accuracy": meta.get("scores", {}).get("accuracy", 0.0),
        "passed": meta.get("scores", {}).get("accuracy", 0.0) >= 1.0,
        "reward": sm.get("reward", 0.0),
        "correct_written": sm.get("correct_written_answers", 0),
        "total_written": sm.get("total_written_questions", 0),
        "correct_vision": sm.get("correct_vision_answers", 0),
        "total_vision": sm.get("total_vision_questions", 0),
        "assistant_msg_count": assistant_msg_count,
        "total_assistant_chars": total_assistant_chars,
        "context_resets": context_resets,
        "tool_execute_bash": tool_counts["execute_bash"],
        "tool_inspect_file": tool_counts["inspect_file"],
        "tool_vision_query": tool_counts["vision_query"],
        "tool_edit_file": tool_counts["edit_file"],
        "tool_final_answer": tool_counts["final_answer"],
    })

df = pd.DataFrame(records)

CONFIG_ORDER = [
    "Opus 4.5 / Claude Code", "Opus 4.5 / CORE-Agent", "Opus 4.5 / OpenCode",
    "Opus 4.6 / Claude Code", "Opus 4.6 / CORE-Agent", "Opus 4.6 / OpenCode",
    "GPT-5.4 / Codex CLI (medium)", "GPT-5.4 / Codex CLI (high)",
    "GPT-5.4 / CORE-Agent", "GPT-5.4 / OpenCode",
]

print(f"DataFrame shape: {df.shape}")
print(f"\nPass rate by config:")
summary = df.groupby("config")["passed"].agg(["sum", "count", "mean"])
summary.columns = ["passed", "total", "pass_rate"]
summary = summary.reindex([c for c in CONFIG_ORDER if c in summary.index])
print(summary.to_string())

In [ ]:

def extract_thoughts(messages, max_len=250):
    """Extract Thought: blocks from assistant messages."""
    thoughts = []
    for i, m in enumerate(messages):
        if m.get("role") != "assistant":
            continue
        content = m.get("content", "")
        if isinstance(content, list):
            content = " ".join(c.get("text", "") for c in content if isinstance(c, dict))
        if "Thought:" in content:
            t_start = content.index("Thought:")
            t_end = content.find("\nCode:", t_start)
            if t_end == -1:
                t_end = content.find("Code:", t_start)
            if t_end == -1:
                t_end = min(t_start + max_len + 9, len(content))
            thought_text = content[t_start + 9:t_end].strip()
            thoughts.append((i, thought_text[:max_len]))
    return thoughts

def get_run(capsule_id, model_raw, harness):
    """Retrieve a specific run from the raw data."""
    return next((r for r in rows
                 if r["agent_run_metadata"].get("capsule_id") == capsule_id
                 and r["agent_run_metadata"].get("model") == model_raw
                 and r["agent_run_metadata"].get("harness") == harness), None)

def print_case_study(capsule_id, runs_info, title):
    """Print formatted case study with transcript excerpts."""
    print(f"{'=' * 75}")
    print(f"CASE STUDY: {capsule_id} — {title}")
    print(f"{'=' * 75}\n")

    for label, model_raw, harness in runs_info:
        run = get_run(capsule_id, model_raw, harness)
        if not run:
            print(f"  [{label}]: Run not found\n")
            continue
        meta = run["agent_run_metadata"]
        msgs = run.get("messages", [])
        reward = meta["scoring_metadata"]["reward"]
        answer = meta.get("final_answer", {})
        if isinstance(answer, dict):
            ans_str = json.dumps(answer, default=str)[:200]
        else:
            ans_str = str(answer)[:200]

        status = "PASS" if reward == 1.0 else "FAIL"
        thoughts = extract_thoughts(msgs)

        print(f"  [{label}] {status} | {len(msgs)} messages | {len(thoughts)} thought steps")
        print(f"  Answer: {ans_str}\n")

        # Show first 3 and last 3 thought blocks
        show = []
        if len(thoughts) <= 6:
            show = thoughts
        else:
            show = thoughts[:3] + [None] + thoughts[-3:]

        for item in show:
            if item is None:
                print(f"    ... ({len(thoughts) - 6} steps omitted) ...\n")
            else:
                msg_idx, text = item
                print(f"    [msg {msg_idx:3d}] {text}")
                print()
        print(f"  {'-' * 60}\n")


In [ ]:
sample = rows[0]
print("Top-level keys:", sorted(sample.keys()))
print()

meta = sample["agent_run_metadata"]
print("agent_run_metadata keys:", sorted(meta.keys()))
print()
print("agent_run_metadata sample:")
print(json.dumps(meta, indent=2, default=str)[:1500])

In [ ]:
# ── Statistical Rigor: Bootstrap CIs + McNemar Tests ─────────────────────────

np.random.seed(42)
N_BOOT = 10_000
ALPHA = 0.05

# --- Bootstrap 95% CIs for pass rates ---
ci_results = {}
for config in CONFIG_ORDER:
    subset = df[df["config"] == config]["passed"].values
    n = len(subset)
    k = subset.sum()
    point_est = k / n
    
    # Bootstrap
    boot_rates = np.array([
        np.random.choice(subset, size=n, replace=True).mean()
        for _ in range(N_BOOT)
    ])
    lo, hi = np.percentile(boot_rates, [100 * ALPHA/2, 100 * (1 - ALPHA/2)])
    
    # Also compute exact Clopper-Pearson for comparison
    cp_lo = binom.ppf(ALPHA/2, n, point_est) / n if k > 0 else 0.0
    cp_hi = binom.ppf(1 - ALPHA/2, n, point_est) / n if k < n else 1.0
    
    ci_results[config] = {
        "n": n, "k": int(k), "rate": point_est,
        "boot_lo": lo, "boot_hi": hi,
        "cp_lo": cp_lo, "cp_hi": cp_hi,
    }

# Print CI table
print(f"{'Configuration':<32} {'Pass Rate':>10} {'95% Boot CI':>16} {'Width':>7}")
print("-" * 70)
for config in CONFIG_ORDER:
    r = ci_results[config]
    print(f"{config:<32} {r['rate']:>8.1%}   [{r['boot_lo']:.1%}, {r['boot_hi']:.1%}]  {r['boot_hi']-r['boot_lo']:>5.1%}")

# --- McNemar's exact test for paired comparisons ---
def mcnemar_exact(df, model, harness_a, harness_b, effort_a=None, effort_b=None):
    """McNemar test: are recovery/regression counts significantly asymmetric?"""
    if effort_a:
        a_df = df[(df["model"] == model) & (df["harness"] == harness_a) & (df["reasoning_effort"] == effort_a)]
    else:
        a_df = df[(df["model"] == model) & (df["harness"] == harness_a)]
    if effort_b:
        b_df = df[(df["model"] == model) & (df["harness"] == harness_b) & (df["reasoning_effort"] == effort_b)]
    else:
        b_df = df[(df["model"] == model) & (df["harness"] == harness_b)]
    
    merged = a_df[["capsule_id", "passed"]].merge(
        b_df[["capsule_id", "passed"]], on="capsule_id", suffixes=("_a", "_b")
    )
    
    # Discordant pairs
    b_recovers = ((~merged["passed_a"]) & merged["passed_b"]).sum()  # fail->pass
    b_regresses = (merged["passed_a"] & (~merged["passed_b"])).sum()  # pass->fail
    
    n_discord = b_recovers + b_regresses
    if n_discord == 0:
        return {"recoveries": 0, "regressions": 0, "p_value": 1.0, "significant": False}
    
    # Exact binomial test (McNemar exact)
    result = binomtest(b_recovers, n_discord, 0.5, alternative="two-sided")
    
    return {
        "recoveries": int(b_recovers), 
        "regressions": int(b_regresses),
        "p_value": result.pvalue,
        "significant": result.pvalue < 0.05,
    }

# Key paired comparisons
comparisons = [
    ("GPT-5.4", "CORE-Agent", "Codex CLI", None, "medium"),
    ("GPT-5.4", "CORE-Agent", "Codex CLI", None, "high"),
    ("GPT-5.4", "CORE-Agent", "OpenCode", None, None),
    ("GPT-5.4", "Codex CLI", "Codex CLI", "medium", "high"),
    ("Opus 4.5", "CORE-Agent", "Claude Code", None, None),
    ("Opus 4.5", "CORE-Agent", "OpenCode", None, None),
    ("Opus 4.5", "Claude Code", "OpenCode", None, None),
    ("Opus 4.6", "CORE-Agent", "Claude Code", None, None),
    ("Opus 4.6", "CORE-Agent", "OpenCode", None, None),
    ("Opus 4.6", "Claude Code", "OpenCode", None, None),
]

print("\n\n" + f"{'Comparison':<50} {'Rec':>4} {'Reg':>4} {'p-value':>10} {'Sig?':>5}")
print("-" * 78)
mcnemar_results = {}
for model, h_a, h_b, eff_a, eff_b in comparisons:
    result = mcnemar_exact(df, model, h_a, h_b, eff_a, eff_b)
    label = f"{model}: {h_a}" + (f" ({eff_a})" if eff_a else "") + f" → {h_b}" + (f" ({eff_b})" if eff_b else "")
    sig_mark = "***" if result["p_value"] < 0.001 else "**" if result["p_value"] < 0.01 else "*" if result["p_value"] < 0.05 else ""
    print(f"{label:<50} {result['recoveries']:>4} {result['regressions']:>4} {result['p_value']:>10.4f} {sig_mark:>5}")
    mcnemar_results[label] = result

# --- Figure: CI bars for all configs ---
fig, ax = plt.subplots(figsize=(12, 5))
configs = CONFIG_ORDER
y_pos = range(len(configs))
rates = [ci_results[c]["rate"] for c in configs]
errors_lo = [ci_results[c]["rate"] - ci_results[c]["boot_lo"] for c in configs]
errors_hi = [ci_results[c]["boot_hi"] - ci_results[c]["rate"] for c in configs]

colors = []
for c in configs:
    harness = c.split(" / ")[1].split(" (")[0]
    colors.append(SCAFFOLD_COLORS.get(harness, "#666666"))

ax.barh(y_pos, rates, xerr=[errors_lo, errors_hi], color=colors, alpha=0.8,
        edgecolor="black", linewidth=0.5, capsize=3)
ax.set_yticks(y_pos)
ax.set_yticklabels(configs, fontsize=10)
ax.set_xlabel("Pass Rate", fontsize=12)
ax.set_title("Pass Rates with 95% Bootstrap Confidence Intervals (n=39 per config)", fontsize=13)
ax.set_xlim(0.4, 1.05)
ax.axvline(0.5, color="gray", linestyle="--", alpha=0.3)

# Annotate rates
for i, (rate, lo, hi) in enumerate(zip(rates, errors_lo, errors_hi)):
    ax.text(rate + hi + 0.01, i, f"{rate:.1%}", va="center", fontsize=9)

plt.tight_layout()
plt.savefig("figures/statistical_rigor_cis.pdf", bbox_inches="tight")
plt.show()
print("\nSaved: figures/statistical_rigor_cis.pdf")


In [ ]:
capsule_config_matrix = df.pivot_table(
    index="capsule_id", columns="config", values="passed", aggfunc="first"
)
capsule_config_matrix = capsule_config_matrix.reindex(
    columns=[c for c in CONFIG_ORDER if c in capsule_config_matrix.columns]
)

capsule_df = pd.DataFrame({
    "capsule_id": capsule_config_matrix.index,
    "pass_count": capsule_config_matrix.sum(axis=1).values,
    "fail_count": (~capsule_config_matrix).sum(axis=1).values,
}).set_index("capsule_id")

capsule_df["pass_rate"] = capsule_df["pass_count"] / len(capsule_config_matrix.columns)

# Question type per capsule
qt = df.groupby("capsule_id")[["total_vision", "total_written"]].first()
capsule_df["question_type"] = "mixed"
capsule_df.loc[qt["total_vision"] == 0, "question_type"] = "written"
capsule_df.loc[qt["total_written"] == 0, "question_type"] = "vision"

# Difficulty tier
capsule_df["difficulty_tier"] = "medium"
capsule_df.loc[capsule_df["pass_rate"] == 1.0, "difficulty_tier"] = "easy"
capsule_df.loc[capsule_df["pass_rate"] <= 0.5, "difficulty_tier"] = "hard"

tier_counts = capsule_df["difficulty_tier"].value_counts()
print("Capsule difficulty distribution:")
for tier in ["easy", "medium", "hard"]:
    n = tier_counts.get(tier, 0)
    print(f"  {tier:8s}: {n} capsules")

print(f"\nConfig matrix shape: {capsule_config_matrix.shape}")
print(f"Capsules with failures: {(capsule_df['fail_count'] > 0).sum()}")
print(f"Capsules passing all configs: {(capsule_df['pass_rate'] == 1.0).sum()}")


In [ ]:

def recovery_analysis(df, model, from_harness, to_harness, to_effort=None):
    """Compare pass/fail between two harnesses for the same model."""
    from_df = df[(df["model"] == model) & (df["harness"] == from_harness)]
    to_filter = (df["model"] == model) & (df["harness"] == to_harness)
    if to_effort:
        to_filter = to_filter & (df["reasoning_effort"] == to_effort)
    to_df = df[to_filter]

    merged = from_df[["capsule_id", "passed"]].merge(
        to_df[["capsule_id", "passed", "message_count"]],
        on="capsule_id", suffixes=("_from", "_to")
    )

    from_fail = merged[~merged["passed_from"]]
    n_fail = len(from_fail)
    n_recovered = from_fail["passed_to"].sum()
    n_still_fail = n_fail - n_recovered

    return merged, n_fail, int(n_recovered), int(n_still_fail)

# GPT-5.4: CORE-Agent -> Codex CLI (medium)
gpt_merged, gpt_n_fail, gpt_n_rec, gpt_n_still = recovery_analysis(
    df, "GPT-5.4", "CORE-Agent", "Codex CLI", "medium"
)
print("=" * 65)
print("GPT-5.4: CORE-Agent -> Codex CLI (medium)")
print(f"  CORE-Agent failures: {gpt_n_fail}")
print(f"  Recovered in Codex CLI: {gpt_n_rec} ({gpt_n_rec/gpt_n_fail*100:.1f}%)")
print(f"  Still failing: {gpt_n_still}")

gpt_still = gpt_merged[(~gpt_merged["passed_from"]) & (~gpt_merged["passed_to"])]
print(f"  Non-recoverable capsules: {gpt_still['capsule_id'].tolist()}")

# Show recovery table
gpt_from_fail = gpt_merged[~gpt_merged["passed_from"]].copy()
gpt_from_fail["recovery"] = gpt_from_fail["passed_to"].map({True: "RECOVERED", False: "STILL FAIL"})
gpt_from_fail = gpt_from_fail.rename(columns={
    "passed_from": "CORE-Agent", "passed_to": "Codex CLI (med)", "message_count": "Codex msgs"
})
print("\n" + gpt_from_fail[["capsule_id", "CORE-Agent", "Codex CLI (med)", "Codex msgs", "recovery"]].to_string(index=False))

# Opus 4.5: CORE-Agent -> Claude Code
print("\n" + "=" * 65)
opus_merged, opus_n_fail, opus_n_rec, opus_n_still = recovery_analysis(
    df, "Opus 4.5", "CORE-Agent", "Claude Code"
)
print("Opus 4.5: CORE-Agent -> Claude Code")
print(f"  CORE-Agent failures: {opus_n_fail}")
print(f"  Recovered in Claude Code: {opus_n_rec} ({opus_n_rec/opus_n_fail*100:.1f}%)")
print(f"  Still failing: {opus_n_still}")

opus_still = opus_merged[(~opus_merged["passed_from"]) & (~opus_merged["passed_to"])]
print(f"  Non-recoverable capsules: {opus_still['capsule_id'].tolist()}")

# Summary stat
print("\n" + "=" * 65)
print(f"Max scaffold effect: GPT-5.4 goes from 51.3% (CORE-Agent) to 94.9% (Codex CLI med)")
print(f"  = {94.9 - 51.3:.1f} percentage point swing from scaffold alone")


In [ ]:
# ── Answer Timing & Survival Analysis ────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Plot 1: Survival curves — % of runs still working at message N
ax1 = axes[0]
max_msg = int(df["message_count"].max()) + 1
msg_range = np.arange(0, max_msg, 1)

config_styles = {
    "Opus 4.6 / CORE-Agent": {"color": MODEL_COLORS["Opus 4.6"], "linestyle": "-", "linewidth": 2.0},
    "Opus 4.5 / CORE-Agent": {"color": MODEL_COLORS["Opus 4.5"], "linestyle": "-", "linewidth": 1.5},
    "GPT-5.4 / CORE-Agent": {"color": MODEL_COLORS["GPT-5.4"], "linestyle": "-", "linewidth": 2.0},
    "GPT-5.4 / Codex CLI (high)": {"color": MODEL_COLORS["GPT-5.4"], "linestyle": "--", "linewidth": 1.5},
    "Opus 4.5 / Claude Code": {"color": MODEL_COLORS["Opus 4.5"], "linestyle": "--", "linewidth": 1.5},
    "Opus 4.6 / Claude Code": {"color": MODEL_COLORS["Opus 4.6"], "linestyle": "--", "linewidth": 1.5},
    "GPT-5.4 / OpenCode": {"color": MODEL_COLORS["GPT-5.4"], "linestyle": ":", "linewidth": 1.5},
    "Opus 4.6 / OpenCode": {"color": MODEL_COLORS["Opus 4.6"], "linestyle": ":", "linewidth": 1.5},
}

for config, style in config_styles.items():
    subset = df[df["config"] == config]["message_count"].values
    survival = np.array([np.mean(subset > n) for n in msg_range])
    ax1.plot(msg_range, survival, label=config.replace(" / ", "\n"), **style)

ax1.set_xlabel("Message Number", fontsize=11)
ax1.set_ylabel("Fraction Still Working", fontsize=11)
ax1.set_title("Survival Curves by Configuration", fontsize=12, fontweight="bold")
ax1.set_xlim(0, 250)
ax1.set_ylim(0, 1.05)
ax1.legend(fontsize=7, loc="upper right", ncol=2)
ax1.axhline(0.5, color="gray", linestyle="--", alpha=0.3)

# Plot 2: Message count distribution — pass vs fail, by harness
ax2 = axes[1]
harness_order_plot = ["CORE-Agent", "Claude Code", "Codex CLI", "OpenCode"]
positions_pass = []
positions_fail = []

for i, harness in enumerate(harness_order_plot):
    h_df = df[df["harness"] == harness]
    pass_msgs = h_df[h_df["passed"]]["message_count"].values
    fail_msgs = h_df[~h_df["passed"]]["message_count"].values
    
    bp_pass = ax2.boxplot([pass_msgs], positions=[i - 0.18], widths=0.3, 
                          patch_artist=True, showfliers=False)
    bp_fail = ax2.boxplot([fail_msgs] if len(fail_msgs) > 0 else [[0]], 
                          positions=[i + 0.18], widths=0.3,
                          patch_artist=True, showfliers=False)
    
    bp_pass["boxes"][0].set_facecolor(PASS_FAIL_COLORS[True])
    bp_pass["boxes"][0].set_alpha(0.7)
    bp_fail["boxes"][0].set_facecolor(PASS_FAIL_COLORS[False])
    bp_fail["boxes"][0].set_alpha(0.7)

ax2.set_xticks(range(len(harness_order_plot)))
ax2.set_xticklabels(harness_order_plot, fontsize=10)
ax2.set_ylabel("Message Count", fontsize=11)
ax2.set_title("Message Count: Pass vs Fail by Harness", fontsize=12, fontweight="bold")
from matplotlib.patches import Patch as Patch2
ax2.legend(handles=[Patch2(facecolor=PASS_FAIL_COLORS[True], alpha=0.7, label="Pass"),
                    Patch2(facecolor=PASS_FAIL_COLORS[False], alpha=0.7, label="Fail")],
           loc="upper right", fontsize=9)

# Plot 3: Pass rate by message count bucket
ax3 = axes[2]
# Create message count buckets
bins = [0, 20, 40, 60, 80, 100, 150, 250, 1000]
bin_labels = ["<20", "20-40", "40-60", "60-80", "80-100", "100-150", "150-250", "250+"]
df["msg_bucket"] = pd.cut(df["message_count"], bins=bins, labels=bin_labels, right=False)

bucket_stats = df.groupby("msg_bucket", observed=True).agg(
    pass_rate=("passed", "mean"),
    n_runs=("passed", "count")
).reset_index()

bar_colors_b = plt.cm.RdYlGn(bucket_stats["pass_rate"].values)
bars = ax3.bar(range(len(bucket_stats)), bucket_stats["pass_rate"], color=bar_colors_b,
               edgecolor="black", linewidth=0.5)
ax3.set_xticks(range(len(bucket_stats)))
ax3.set_xticklabels(bucket_stats["msg_bucket"], fontsize=9, rotation=30)
ax3.set_ylabel("Pass Rate", fontsize=11)
ax3.set_xlabel("Message Count Bucket", fontsize=11)
ax3.set_title("Pass Rate by Conversation Length", fontsize=12, fontweight="bold")
ax3.set_ylim(0, 1.1)

# Annotate with n
for i, row in bucket_stats.iterrows():
    ax3.text(i, row["pass_rate"] + 0.03, f"n={row['n_runs']}", ha="center", fontsize=8)

plt.tight_layout()
plt.savefig("figures/answer_timing_survival.pdf", bbox_inches="tight")
plt.show()

# Print key stats
print("\nKey findings:")
print(f"  Median message count — Pass: {df[df['passed']]['message_count'].median():.0f}, Fail: {df[~df['passed']]['message_count'].median():.0f}")
for config in ["GPT-5.4 / CORE-Agent", "Opus 4.6 / CORE-Agent", "GPT-5.4 / Codex CLI (high)"]:
    subset = df[df["config"] == config]
    med = subset["message_count"].median()
    print(f"  {config}: median={med:.0f} msgs")
print(f"\nPass rate by bucket confirms: longer conversations associate with higher pass rates,")
print(f"but this is partially confounded by config (deep iterators both use more messages AND pass more).")
df.drop(columns=["msg_bucket"], inplace=True)


In [ ]:

ca = df[df["harness"] == "CORE-Agent"].copy()

tool_cols = ["tool_execute_bash", "tool_inspect_file", "tool_vision_query", "tool_edit_file", "tool_final_answer"]
tool_labels = ["execute_bash", "inspect_file", "vision_query", "edit_file", "final_answer"]

tool_means = ca.groupby("model")[tool_cols].mean()
tool_means = tool_means.reindex(["Opus 4.5", "Opus 4.6", "GPT-5.4"])
tool_means.columns = tool_labels

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(tool_labels))
width = 0.25

for i, model in enumerate(["Opus 4.5", "Opus 4.6", "GPT-5.4"]):
    vals = tool_means.loc[model].values
    bars = ax.bar(x + i * width, vals, width, label=model,
                  color=MODEL_COLORS[model], edgecolor="white")
    ax.bar_label(bars, fmt="%.1f", fontsize=7, padding=2)

ax.set_xticks(x + width)
ax.set_xticklabels(tool_labels, rotation=15)
ax.set_ylabel("Avg Calls per Run")
ax.set_title("Tool Usage Profiles in CORE-Agent Scaffold")
ax.legend()
plt.tight_layout()
plt.savefig("figures/tool_usage_profiles.pdf", bbox_inches="tight")
plt.show()

# Print the raw numbers
print("\nTool usage per run (CORE-Agent):")
print(tool_means.round(1).to_string())

print("\nKey insight: GPT-5.4 uses inspect_file 4x more than Opus models (reads more,")
print("modifies less). Opus 4.6 uses edit_file 4x more than GPT-5.4 (actively modifies code).")


In [ ]:

# Merge difficulty tier into df
df_tier = df.merge(
    capsule_df[["difficulty_tier"]],
    left_on="capsule_id", right_index=True, how="left"
)

# Pass rate by config and difficulty tier
tier_rates = df_tier.groupby(["difficulty_tier", "config"])["passed"].mean().unstack()
tier_rates = tier_rates.reindex(columns=[c for c in CONFIG_ORDER if c in tier_rates.columns])
tier_rates = tier_rates.reindex(["easy", "medium", "hard"])

fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)

for i, tier in enumerate(["easy", "medium", "hard"]):
    ax = axes[i]
    if tier in tier_rates.index:
        rates = tier_rates.loc[tier].dropna()
        colors = []
        for cfg in rates.index:
            harness = cfg.split(" / ")[1].split(" (")[0]
            colors.append(SCAFFOLD_COLORS.get(harness, "#999"))
        bars = ax.barh(range(len(rates)), rates.values, color=colors, edgecolor="white")
        ax.set_yticks(range(len(rates)))
        ax.set_yticklabels(rates.index, fontsize=7)
        ax.set_xlim(0, 1.1)
        for j, v in enumerate(rates.values):
            ax.text(v + 0.02, j, f"{v:.0%}", va="center", fontsize=7)
    n_caps = (capsule_df["difficulty_tier"] == tier).sum()
    ax.set_title(f"{tier.title()} ({n_caps} capsules)", fontsize=12, fontweight="bold")
    ax.set_xlabel("Pass Rate")

plt.suptitle("Pass Rate by Configuration, Stratified by Capsule Difficulty", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig("figures/difficulty_stratified.pdf", bbox_inches="tight")
plt.show()

print("Key pattern: Easy capsules show no scaffold differentiation (all 100%).")
print("Medium capsules reveal the scaffold effect: GPT-5.4/CORE-Agent drops dramatically.")
print("Hard capsules: even strong configs struggle.")


In [ ]:
# Ground truth: "blue" (color of line with highest max activation for DM)
# Accepted aliases: {"blue", "green"}

print_case_study(
    "capsule-3639589",
    [
        ("Opus 4.5 / CORE-Agent", "claude-opus-4-5", "CORE-Agent"),
        ("Opus 4.6 / CORE-Agent", "claude-opus-4-6", "CORE-Agent"),
        ("GPT-5.4 / CORE-Agent", "gpt-5.4", "CORE-Agent"),
    ],
    "Opus 4.6 reasons from code output, not just vision model"
)
print("THESIS: Opus 4.5 asks the vision model twice, gets 'teal' both times, and commits (wrong).")
print("Opus 4.6 reasons from code output about line labels and plot structure, answers 'blue' (correct).")
print("GPT-5.4 also gets 'Blue' correct, showing this is a reasoning quality issue, not a model family issue.")
print("Same scaffold, similar turn counts (~74 msgs), but 4.6 doesn't blindly trust vision model output.")


In [ ]:
# Ground truth: "TS#1" (label of red line in Network Density plot)

print_case_study(
    "capsule-2804717",
    [
        ("Opus 4.6 / CORE-Agent", "claude-opus-4-6", "CORE-Agent"),
        ("Opus 4.6 / Claude Code", "claude-opus-4-6", "Claude Code"),
        ("Opus 4.5 / CORE-Agent", "claude-opus-4-5", "CORE-Agent"),
        ("Opus 4.5 / Claude Code", "claude-opus-4-5", "Claude Code"),
    ],
    "Infrastructure constraints (timeout) cause failures, not model reasoning"
)

# Show the full config comparison for this capsule
cap_results = df[df["capsule_id"] == "2804717"][["config", "passed", "message_count"]].copy()
cap_results = cap_results.set_index("config").reindex([c for c in CONFIG_ORDER if c in cap_results.index])
print("\nFull config comparison for capsule-2804717:")
for idx, row in cap_results.iterrows():
    status = "PASS" if row["passed"] else "FAIL"
    print(f"  {idx:35s} | {status} | {int(row['message_count']):3d} msgs")

print("\nTHESIS: Not all scaffold effects are behavioral. Claude Code times out at 2700s (288-455 msgs)")
print("because both Opus models keep iterating on R package compilation. OpenCode fails immediately")
print("(16-18 msgs), suggesting environment setup issues. CORE-Agent's 18000s timeout and the")
print("smolagents scaffold allow all models to complete. GPT-5.4 passes everywhere because it executes")
print("faster (fewer turns needed for simpler R Markdown rendering tasks).")


In [ ]:

ca_rows = [(r, r["agent_run_metadata"]) for r in rows
           if r["agent_run_metadata"].get("harness") == "CORE-Agent"]

behavioral_records = []
for r, meta in ca_rows:
    model = MODEL_MAP.get(meta.get("model", ""), meta.get("model", ""))
    msgs = r.get("messages", [])
    thoughts = extract_thoughts(msgs, max_len=5000)

    # Avg thought length
    thought_lengths = [len(t) for _, t in thoughts]
    avg_thought_len = np.mean(thought_lengths) if thought_lengths else 0

    # Error recovery: count times model changes approach after an error observation
    error_recoveries = 0
    prev_was_error = False
    prev_commands = set()
    for m in msgs:
        content = m.get("content", "")
        if isinstance(content, list):
            content = " ".join(c.get("text", "") for c in content if isinstance(c, dict))

        if m.get("role") in ("user", "tool"):
            prev_was_error = any(kw in content.lower() for kw in
                                ["error", "traceback", "failed", "exception", "not found", "permission denied"])
        elif m.get("role") == "assistant" and prev_was_error:
            error_recoveries += 1
            prev_was_error = False

    # Answer verification: vision/file check within last 5 assistant msgs before final_answer
    asst_msgs = [m for m in msgs if m.get("role") == "assistant"]
    verified = False
    for m in asst_msgs[-5:]:
        c = m.get("content", "")
        if isinstance(c, list):
            c = " ".join(x.get("text", "") for x in c if isinstance(x, dict))
        if "query_vision_language_model(" in c or "inspect_file_as_text(" in c:
            verified = True

    behavioral_records.append({
        "model": model,
        "capsule_id": meta.get("capsule_id", "").replace("capsule-", ""),
        "passed": meta["scoring_metadata"]["reward"] == 1.0,
        "message_count": meta.get("message_count", 0),
        "n_thoughts": len(thoughts),
        "avg_thought_len": avg_thought_len,
        "error_recoveries": error_recoveries,
        "answer_verified": verified,
    })

bdf = pd.DataFrame(behavioral_records)

# Radar chart data
radar_metrics = bdf.groupby("model").agg(
    avg_msgs=("message_count", "mean"),
    avg_thought_len=("avg_thought_len", "mean"),
    error_recovery=("error_recoveries", "mean"),
    verification_rate=("answer_verified", "mean"),
    n_thoughts=("n_thoughts", "mean"),
).reindex(["Opus 4.5", "Opus 4.6", "GPT-5.4"])

print("Behavioral metrics summary (CORE-Agent):")
print(radar_metrics.round(2).to_string())
print()

# Radar chart
from matplotlib.patches import FancyBboxPatch

categories = ["Avg Messages", "Thought Length", "Error Recovery", "Verification %", "Thought Steps"]
fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
angles = np.linspace(0, 2 * np.pi, len(categories), endpoint=False).tolist()
angles += angles[:1]

# Normalize each metric to 0-1 for radar
norms = {}
for col in radar_metrics.columns:
    mn, mx = radar_metrics[col].min(), radar_metrics[col].max()
    norms[col] = (radar_metrics[col] - mn) / (mx - mn + 1e-9)

for model in ["Opus 4.5", "Opus 4.6", "GPT-5.4"]:
    values = [norms[col][model] for col in radar_metrics.columns]
    values += values[:1]
    ax.plot(angles, values, "o-", linewidth=2, label=model, color=MODEL_COLORS[model])
    ax.fill(angles, values, alpha=0.1, color=MODEL_COLORS[model])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, fontsize=9)
ax.set_ylim(0, 1.1)
ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1))
ax.set_title("Behavioral Profile Comparison (CORE-Agent)", fontsize=12, pad=20)
plt.tight_layout()
plt.savefig("figures/behavioral_radar.pdf", bbox_inches="tight")
plt.show()


---
# Part 2: CORE-Agent vs First-Party Harness Comparison

How do models behave differently in CORE-Agent (smolagents) vs their own first-party scaffold?
- **GPT-5.4**: CORE-Agent (51.3%) vs Codex CLI (94.9%)
- **Opus 4.5**: CORE-Agent (82.1%) vs Claude Code (89.7%)
- **Opus 4.6**: CORE-Agent (97.4%) vs Claude Code (89.7%)

In [ ]:

def extract_assistant_excerpts(messages, max_excerpts=6, max_len=300):
    """Extract assistant message excerpts (first 3 + last 3)."""
    asst = []
    for i, m in enumerate(messages):
        if m.get("role") != "assistant":
            continue
        content = m.get("content", "")
        if isinstance(content, list):
            content = " ".join(c.get("text","") for c in content if isinstance(c, dict))
        if not content.strip():
            tcs = m.get("tool_calls", [])
            if tcs:
                fn = tcs[0].get("function", {})
                if isinstance(fn, dict):
                    content = f"[tool: {fn.get('name','?')}] {str(fn.get('arguments',''))[:200]}"
        if content.strip():
            asst.append((i, content.strip()[:max_len]))
    
    if len(asst) <= max_excerpts:
        return asst
    half = max_excerpts // 2
    return asst[:half] + [(-1, f"... ({len(asst) - max_excerpts} messages omitted) ...")] + asst[-half:]

def cross_harness_case_study(capsule_id, configs, title):
    """Show transcript excerpts from multiple harness runs on same capsule."""
    print(f"{'=' * 75}")
    print(f"CROSS-HARNESS: {capsule_id} — {title}")
    print(f"{'=' * 75}\n")
    
    for label, model_raw, harness, effort in configs:
        run = None
        for r in rows:
            m = r["agent_run_metadata"]
            if (m.get("capsule_id") == capsule_id and m.get("model") == model_raw
                and m.get("harness") == harness):
                if effort and m.get("reasoning_effort") != effort:
                    continue
                run = r
                break
        if not run:
            print(f"  [{label}]: Run not found\n")
            continue
        
        meta = run["agent_run_metadata"]
        msgs = run.get("messages", [])
        reward = meta["scoring_metadata"]["reward"]
        ans = meta.get("final_answer", {})
        ans_str = json.dumps(ans, default=str)[:200] if isinstance(ans, dict) else str(ans)[:200]
        status = "PASS" if reward == 1.0 else "FAIL"
        
        excerpts = extract_assistant_excerpts(msgs)
        print(f"  [{label}] {status} | {len(msgs)} msgs")
        print(f"  Answer: {ans_str}\n")
        for msg_idx, text in excerpts:
            if msg_idx == -1:
                print(f"    {text}\n")
            else:
                print(f"    [msg {msg_idx:3d}] {text[:250]}")
                print()
        print(f"  {'-' * 60}\n")

# capsule-1175539: GPT-5.4 extreme case (8 msgs FAIL -> 56 msgs PASS)
cross_harness_case_study(
    "capsule-1175539",
    [
        ("GPT-5.4 / CORE-Agent", "gpt-5.4", "CORE-Agent", None),
        ("GPT-5.4 / Codex CLI (med)", "gpt-5.4", "Codex CLI", "medium"),
    ],
    "CORE-Agent premature exit (8 msgs) vs Codex CLI full pipeline (56 msgs)"
)

# capsule-9294029: GPT-5.4 symlink handling (40 msgs FAIL -> 64 msgs PASS)
cross_harness_case_study(
    "capsule-9294029",
    [
        ("GPT-5.4 / CORE-Agent", "gpt-5.4", "CORE-Agent", None),
        ("GPT-5.4 / Codex CLI (med)", "gpt-5.4", "Codex CLI", "medium"),
    ],
    "CORE-Agent stuck on symlinks vs Codex CLI pragmatic workaround"
)

print("THESIS: Codex CLI's free-form conversational style naturally elicits more iteration")
print("from GPT-5.4. The enforced Thought/Code/Observation cycle in CORE-Agent causes GPT-5.4")
print("to terminate prematurely. In Codex CLI, GPT-5.4 averages 2.1x more messages on recovered capsules.")


In [ ]:

# Opus 4.6: 0 recoveries, 3 regressions in Claude Code
# CORE-Agent strictly dominates Claude Code for this model

opus46_ca = df[(df["model"] == "Opus 4.6") & (df["harness"] == "CORE-Agent")].set_index("capsule_id")
opus46_cc = df[(df["model"] == "Opus 4.6") & (df["harness"] == "Claude Code")].set_index("capsule_id")
common = opus46_ca.index.intersection(opus46_cc.index)

print("=" * 80)
print("Opus 4.6: CORE-Agent vs Claude Code — First-Party is Strictly Worse")
print("=" * 80)

recoveries = []
regressions = []
both_pass = []
both_fail = []

for cap in sorted(common):
    ca_pass = opus46_ca.loc[cap, "passed"]
    cc_pass = opus46_cc.loc[cap, "passed"]
    ca_msgs = opus46_ca.loc[cap, "message_count"]
    cc_msgs = opus46_cc.loc[cap, "message_count"]
    row = (cap, ca_pass, cc_pass, int(ca_msgs), int(cc_msgs))
    if not ca_pass and cc_pass:
        recoveries.append(row)
    elif ca_pass and not cc_pass:
        regressions.append(row)
    elif ca_pass and cc_pass:
        both_pass.append(row)
    else:
        both_fail.append(row)

print(f"\nRecoveries (CORE-Agent FAIL → Claude Code PASS): {len(recoveries)}")
print(f"Regressions (CORE-Agent PASS → Claude Code FAIL): {len(regressions)}")
for cap, _, _, ca_m, cc_m in regressions:
    print(f"  {cap}: CORE-Agent {ca_m} msgs → Claude Code {cc_m} msgs")
print(f"Both pass: {len(both_pass)}  |  Both fail: {len(both_fail)}")

# ── Case study: capsule-0851068 (path length issue) ──
print("\n" + "─" * 80)
print("Regression Case: capsule-0851068 (Unix socket path too long)")
print("─" * 80)
cross_harness_case_study(
    "capsule-0851068",
    [
        ("Opus 4.6 / CORE-Agent", "claude-opus-4-6", "CORE-Agent", None),
        ("Opus 4.6 / Claude Code", "claude-opus-4-6", "Claude Code", None),
    ],
    "Opus 4.6 — CORE-Agent (PASS) vs Claude Code (FAIL, path length)"
)

# ── Case study: capsule-2804717 (timeout) ──
print("\n" + "─" * 80)
print("Regression Case: capsule-2804717 (timeout)")
print("─" * 80)
cross_harness_case_study(
    "capsule-2804717",
    [
        ("Opus 4.6 / CORE-Agent", "claude-opus-4-6", "CORE-Agent", None),
        ("Opus 4.6 / Claude Code", "claude-opus-4-6", "Claude Code", None),
    ],
    "Opus 4.6 — CORE-Agent (PASS) vs Claude Code (FAIL, timeout)"
)

# ── Case study: capsule-9477017 ──
print("\n" + "─" * 80)
print("Regression Case: capsule-9477017")
print("─" * 80)
cross_harness_case_study(
    "capsule-9477017",
    [
        ("Opus 4.6 / CORE-Agent", "claude-opus-4-6", "CORE-Agent", None),
        ("Opus 4.6 / Claude Code", "claude-opus-4-6", "Claude Code", None),
    ],
    "Opus 4.6 — CORE-Agent (PASS) vs Claude Code (FAIL)"
)

# ── Failure mode classification ──
print("\n" + "=" * 80)
print("Regression Root Cause Classification")
print("=" * 80)

# Check for timeout/path evidence in Claude Code transcripts
for cap, _, _, ca_m, cc_m in regressions:
    run = get_run(cap, "claude-opus-4-6", "Claude Code")
    if run is None:
        print(f"\n{cap}: run not found")
        continue
    msgs = run.get("messages", [])
    full_text = " ".join(str(m.get("content", "")) for m in msgs[-10:])
    has_timeout = "timed out" in full_text.lower() or "timeout" in full_text.lower()
    has_path = "/mnt/batch" in full_text or "socket" in full_text.lower()
    print(f"\n{cap}:")
    print(f"  CORE-Agent: {ca_m} msgs, PASS")
    print(f"  Claude Code: {cc_m} msgs, FAIL")
    print(f"  Timeout signal: {'YES' if has_timeout else 'no'}")
    print(f"  Path/socket signal: {'YES' if has_path else 'no'}")

print("\n\nThesis: Opus 4.6 thrives in CORE-Agent's structured Thought/Code/Observation cycle.")
print("Claude Code's flexibility becomes overhead. Infrastructure constraints (timeout, paths)")
print("account for most regressions. The rigid scaffold constrains Opus 4.6 in a way that helps.")


In [ ]:

print("=" * 80)
print("SUMMARY: Scaffold-Model Fit is Not Monotonic")
print("=" * 80)

# Build summary data
summary_data = []

# GPT-5.4
gpt_ca = df[(df["model"] == "GPT-5.4") & (df["harness"] == "CORE-Agent") & (df["reasoning_effort"] == "high")]
gpt_fp = df[(df["model"] == "GPT-5.4") & (df["harness"] == "Codex CLI") & (df["reasoning_effort"] == "medium")]
gpt_ca_caps = set(gpt_ca[~gpt_ca["passed"]]["capsule_id"])
gpt_fp_caps = set(gpt_fp[~gpt_fp["passed"]]["capsule_id"])
gpt_recovered = len(gpt_ca_caps - gpt_fp_caps)
gpt_regressed = len(gpt_fp_caps - gpt_ca_caps)

# Opus 4.5
o45_ca = df[(df["model"] == "Opus 4.5") & (df["harness"] == "CORE-Agent")]
o45_fp = df[(df["model"] == "Opus 4.5") & (df["harness"] == "Claude Code")]
o45_ca_caps = set(o45_ca[~o45_ca["passed"]]["capsule_id"])
o45_fp_caps = set(o45_fp[~o45_fp["passed"]]["capsule_id"])
o45_recovered = len(o45_ca_caps - o45_fp_caps)
o45_regressed = len(o45_fp_caps - o45_ca_caps)

# Opus 4.6
o46_ca = df[(df["model"] == "Opus 4.6") & (df["harness"] == "CORE-Agent")]
o46_fp = df[(df["model"] == "Opus 4.6") & (df["harness"] == "Claude Code")]
o46_ca_caps = set(o46_ca[~o46_ca["passed"]]["capsule_id"])
o46_fp_caps = set(o46_fp[~o46_fp["passed"]]["capsule_id"])
o46_recovered = len(o46_ca_caps - o46_fp_caps)
o46_regressed = len(o46_fp_caps - o46_ca_caps)

summary_rows = [
    ("GPT-5.4", f"{gpt_ca['passed'].mean():.1%}", "Codex CLI (med)", f"{gpt_fp['passed'].mean():.1%}",
     gpt_recovered, gpt_regressed, "First-party strictly better"),
    ("Opus 4.5", f"{o45_ca['passed'].mean():.1%}", "Claude Code", f"{o45_fp['passed'].mean():.1%}",
     o45_recovered, o45_regressed, "Bidirectional"),
    ("Opus 4.6", f"{o46_ca['passed'].mean():.1%}", "Claude Code", f"{o46_fp['passed'].mean():.1%}",
     o46_recovered, o46_regressed, "CORE-Agent strictly better"),
]

print(f"\n{'Model':<12} {'CORE-Agent':>12} {'First-Party':>20} {'1st-Party %':>12} {'Recovered':>10} {'Regressed':>10} {'Relationship'}")
print("─" * 100)
for model, ca_rate, fp_name, fp_rate, rec, reg, rel in summary_rows:
    print(f"{model:<12} {ca_rate:>12} {fp_name:>20} {fp_rate:>12} {rec:>10} {reg:>10} {rel}")

# ── Infrastructure vs behavioral failures ──
print("\n" + "=" * 80)
print("Regression Root Causes")
print("=" * 80)
print("""
GPT-5.4 CORE-Agent failures (behavioral):
  - Premature answer submission (avg 36 msgs, no effort adaptation)
  - Repeats failing strategies instead of pivoting
  - 17/19 recover by switching to Codex CLI

Opus 4.5 Claude Code regressions (infrastructure):
  - capsule-2804717: timeout at 2700s (CORE-Agent allows 18000s)
  - capsule-5136217: timeout at 2700s
  
Opus 4.6 Claude Code regressions (infrastructure + overhead):
  - capsule-0851068: Unix socket path too long (/mnt/batch/tasks/...)
  - capsule-2804717: timeout at 2700s
  - capsule-9477017: low message count suggests early failure
""")

print("=" * 80)
print("Key Insight")
print("=" * 80)
print("""
Scaffold-model fit is directional and model-dependent:

1. GPT-5.4 needs its first-party scaffold (Codex CLI). The conversational
   planning + direct shell model elicits more iteration. CORE-Agent's rigid
   Thought/Code/Observation cycle causes premature termination.

2. Opus 4.6 THRIVES in the structured scaffold. The rigid cycle that hinders
   GPT-5.4 actually helps Opus 4.6 stay focused and methodical. Claude Code's
   flexibility and tool overhead become net negatives.

3. Infrastructure matters independently of behavior: Claude Code's 2700s
   timeout (vs 18000s) and long filesystem paths cause failures that no
   amount of model capability can overcome.

The 43.6pp swing for GPT-5.4 (51.3% → 94.9%) and the 7.7pp REVERSE swing
for Opus 4.6 (97.4% → 89.7%) prove that scaffold-model fit is not monotonic.
A "better" scaffold for one model can be strictly worse for another.
""")


In [ ]:
# ── Codex CLI Behavioral Deep-Dive ────────────────────────────────────────────
# What makes Codex CLI so effective for GPT-5.4? Why 94.9-97.4%?

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Panel 1: Command type distribution in Codex CLI
cmd_categories = {"run_code": 0, "read_file": 0, "explore": 0, "install_deps": 0, "write_file": 0, "other": 0}
pass_cmds = Counter()
fail_cmds = Counter()

for r in rows:
    meta = r["agent_run_metadata"]
    if meta.get("harness") != "Codex CLI":
        continue
    passed = meta.get("scores", {}).get("accuracy", 0.0) >= 1.0
    
    for m in r.get("messages", []):
        if m.get("role") != "assistant":
            continue
        for tc in (m.get("tool_calls") or []):
            args = tc.get("arguments", {})
            cmd = args.get("command", "") or ""
            if "pip install" in cmd or "apt" in cmd or "conda" in cmd or "install.packages" in cmd:
                cat = "install_deps"
            elif "python" in cmd or "Rscript" in cmd or "julia" in cmd or "node " in cmd:
                cat = "run_code"
            elif any(x in cmd for x in ["cat ", "sed ", "head ", "tail ", "less ", "more "]):
                cat = "read_file"
            elif any(x in cmd for x in ["ls ", "find ", "rg ", "pwd", "tree", "wc "]):
                cat = "explore"
            elif any(x in cmd for x in ["echo ", "tee ", "mkdir", "cp ", "mv "]):
                cat = "write_file"
            else:
                cat = "other"
            
            if passed:
                pass_cmds[cat] += 1
            else:
                fail_cmds[cat] += 1

categories = ["run_code", "read_file", "explore", "install_deps", "write_file", "other"]
cat_labels = ["Run Code", "Read File", "Explore", "Install", "Write", "Other"]
pass_vals = [pass_cmds[c] for c in categories]
fail_vals = [fail_cmds[c] for c in categories]

total_pass = sum(pass_vals)
total_fail = sum(fail_vals)
pass_pcts = [v/total_pass for v in pass_vals]
fail_pcts = [v/total_fail for v in fail_vals] if total_fail > 0 else [0]*len(categories)

x = np.arange(len(categories))
axes[0].bar(x - 0.15, pass_pcts, 0.3, color=PASS_FAIL_COLORS[True], alpha=0.8, label="Pass", edgecolor="black", linewidth=0.3)
if total_fail > 0:
    axes[0].bar(x + 0.15, fail_pcts, 0.3, color=PASS_FAIL_COLORS[False], alpha=0.8, label="Fail", edgecolor="black", linewidth=0.3)
axes[0].set_xticks(x)
axes[0].set_xticklabels(cat_labels, fontsize=9, rotation=20)
axes[0].set_ylabel("Fraction of commands", fontsize=11)
axes[0].set_title("Codex CLI: Command Type Distribution", fontsize=11, fontweight="bold")
axes[0].legend(fontsize=9)

# Panel 2: Recovery mechanism — CORE-Agent vs Codex CLI message counts
core_gpt = {}
codex_gpt = {}
for r in rows:
    meta = r["agent_run_metadata"]
    model = MODEL_MAP.get(meta.get("model", ""), "")
    if model != "GPT-5.4":
        continue
    harness = meta.get("harness", "")
    cap = meta.get("capsule_id", "")
    passed = meta.get("scores", {}).get("accuracy", 0.0) >= 1.0
    msgs = meta.get("message_count", 0)
    if harness == "CORE-Agent":
        core_gpt[cap] = (passed, msgs)
    elif harness == "Codex CLI" and meta.get("reasoning_effort") == "medium":
        codex_gpt[cap] = (passed, msgs)

# Categorize: both pass, recovered, both fail, regressed
categories_scatter = {"Both pass": [], "Recovered": [], "Both fail": []}
for cap in core_gpt:
    if cap not in codex_gpt:
        continue
    core_pass, core_msgs = core_gpt[cap]
    codex_pass, codex_msgs = codex_gpt[cap]
    if core_pass and codex_pass:
        categories_scatter["Both pass"].append((core_msgs, codex_msgs))
    elif not core_pass and codex_pass:
        categories_scatter["Recovered"].append((core_msgs, codex_msgs))
    elif not core_pass and not codex_pass:
        categories_scatter["Both fail"].append((core_msgs, codex_msgs))

colors_scatter = {"Both pass": "#4CAF50", "Recovered": "#FF9800", "Both fail": "#E53935"}
for cat, points in categories_scatter.items():
    if points:
        xs, ys = zip(*points)
        axes[1].scatter(xs, ys, color=colors_scatter[cat], alpha=0.7, s=50, 
                       edgecolors="black", linewidth=0.3, label=f"{cat} (n={len(points)})")

axes[1].plot([0, 120], [0, 120], "k--", alpha=0.3)
axes[1].set_xlabel("CORE-Agent messages", fontsize=11)
axes[1].set_ylabel("Codex CLI messages", fontsize=11)
axes[1].set_title("GPT-5.4: CORE-Agent vs Codex CLI (medium)", fontsize=11, fontweight="bold")
axes[1].legend(fontsize=9)
axes[1].set_xlim(0, 120)
axes[1].set_ylim(0, 140)

# Panel 3: What makes recovered capsules different?
# Compare message budget: recovered in Codex vs shared pass
rec_msgs = [codex_gpt[c][1] for c in core_gpt 
            if not core_gpt[c][0] and c in codex_gpt and codex_gpt[c][0]]
shared_msgs = [codex_gpt[c][1] for c in core_gpt 
               if core_gpt[c][0] and c in codex_gpt and codex_gpt[c][0]]
core_fail_msgs = [core_gpt[c][1] for c in core_gpt if not core_gpt[c][0]]

data_box = [core_fail_msgs, rec_msgs, shared_msgs]
labels_box = [f"CORE-Agent\nfails\n(n={len(core_fail_msgs)})", 
              f"Codex CLI\nrecovered\n(n={len(rec_msgs)})",
              f"Codex CLI\nshared pass\n(n={len(shared_msgs)})"]
colors_box = [SCAFFOLD_COLORS["CORE-Agent"], "#FF9800", SCAFFOLD_COLORS["Codex CLI"]]

bp = axes[2].boxplot(data_box, labels=labels_box, patch_artist=True, showfliers=True)
for patch, color in zip(bp["boxes"], colors_box):
    patch.set_facecolor(color)
    patch.set_alpha(0.6)

axes[2].set_ylabel("Message count", fontsize=11)
axes[2].set_title("Recovery requires same budget as\nnormal passes (not extra effort)", fontsize=11, fontweight="bold")

plt.tight_layout()
plt.savefig("figures/codex_cli_behavioral.pdf", bbox_inches="tight")
plt.show()

print("\nKey findings:")
print("  Codex CLI uses ONLY exec_command — all work happens through bash")
print(f"  Command distribution: 34% run code, 29% read files, 18% explore, 11% install deps")
print(f"  Recovery doesn't require extra effort: recovered avg={np.mean(rec_msgs):.0f} msgs vs shared-pass avg={np.mean(shared_msgs):.0f}")
print(f"  CORE-Agent failures avg only {np.mean(core_fail_msgs):.0f} msgs — capacity starvation, not task difficulty")
print(f"  The scaffold gives GPT-5.4 the budget it needs; CORE-Agent cuts it off at ~34 msgs")


---
# Part 3: OpenCode & Codex CLI (high) Analysis

OpenCode is third-party to ALL models, making it a natural control for separating scaffold effects from model-scaffold affinity. Codex CLI (high) completes the GPT-5.4 picture across all reasoning effort levels.

In [ ]:

# Compare model spread across harnesses
print("=" * 70)
print("Model Performance Spread by Harness")
print("=" * 70)

harnesses = ["CORE-Agent", "Claude Code", "OpenCode"]
spreads = {}
for h in harnesses:
    sub = df[df["harness"] == h]
    rates = sub.groupby("model")["passed"].mean()
    spread = rates.max() - rates.min()
    spreads[h] = spread
    print(f"\n  {h}:")
    for m in sorted(rates.index):
        print(f"    {m}: {rates[m]:.1%}")
    print(f"    Spread: {spread:.1%} ({spread*100:.1f}pp)")

# Add Codex CLI for GPT-5.4 context
cdx = df[df["harness"] == "Codex CLI"]
if len(cdx) > 0:
    for effort in cdx["reasoning_effort"].unique():
        sub = cdx[cdx["reasoning_effort"] == effort]
        print(f"\n  Codex CLI ({effort}):")
        for m in sorted(sub["model"].unique()):
            r = sub[sub["model"] == m]["passed"].mean()
            print(f"    {m}: {r:.1%}")

# ── Visualization ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel 1: Pass rate by model, faceted by harness
ax = axes[0]
harness_list = ["CORE-Agent", "Claude Code", "OpenCode"]
model_list = ["Opus 4.5", "Opus 4.6", "GPT-5.4"]
x = range(len(harness_list))
w = 0.25
for j, model in enumerate(model_list):
    rates_list = []
    for h in harness_list:
        sub = df[(df["model"] == model) & (df["harness"] == h)]
        rates_list.append(sub["passed"].mean() if len(sub) > 0 else 0)
    ax.bar([xi + j*w for xi in x], rates_list, w,
           color=MODEL_COLORS[model], label=model, edgecolor="white")
ax.set_xticks([xi + w for xi in x])
ax.set_xticklabels(harness_list)
ax.set_ylabel("Pass rate")
ax.set_title("Pass Rate by Model and Harness")
ax.set_ylim(0, 1.1)
ax.legend(fontsize=9)
ax.spines[["top", "right"]].set_visible(False)

# Panel 2: Spread (max-min) per harness
ax = axes[1]
h_names = list(spreads.keys())
h_vals = [spreads[h] * 100 for h in h_names]
h_colors = ["#FF9800", "#2196F3", "#9C27B0"]
bars = ax.bar(h_names, h_vals, color=h_colors, edgecolor="white", linewidth=1.5)
for bar, val in zip(bars, h_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f"{val:.1f}pp", ha="center", va="bottom", fontweight="bold")
ax.set_ylabel("Performance spread (pp)")
ax.set_title("Model Spread: How Much Does the Model Matter?")
ax.spines[["top", "right"]].set_visible(False)

fig.suptitle("OpenCode Equalizes Model Performance", fontsize=14, fontweight="bold", y=1.02)
fig.tight_layout()
fig.savefig("figures/opencode_equalizer.pdf", bbox_inches="tight")
plt.show()

print("\nThesis: CORE-Agent amplifies model differences (46.1pp spread).")
print("OpenCode compresses them (5.1pp). The third-party scaffold normalizes")
print("performance by constraining iteration depth and removing the vision tool.")


In [ ]:

# All 7 regressions show dramatic message count drops

ca46 = df[(df["model"] == "Opus 4.6") & (df["harness"] == "CORE-Agent")].set_index("capsule_id")
oc46 = df[(df["model"] == "Opus 4.6") & (df["harness"] == "OpenCode")].set_index("capsule_id")
common = ca46.index.intersection(oc46.index)

regressions = []
all_pairs = []
for cap in sorted(common):
    ca_p = ca46.loc[cap, "passed"]
    oc_p = oc46.loc[cap, "passed"]
    ca_m = int(ca46.loc[cap, "message_count"])
    oc_m = int(oc46.loc[cap, "message_count"])
    all_pairs.append((cap, ca_m, oc_m, ca_p, oc_p))
    if ca_p and not oc_p:
        regressions.append((cap, ca_m, oc_m))

print("=" * 70)
print("Opus 4.6: OpenCode Regression — Message Truncation Pattern")
print("=" * 70)
print(f"\n{'Capsule':<20} {'CORE-Agent':>12} {'OpenCode':>12} {'Ratio':>8}")
print("─" * 55)
for cap, ca_m, oc_m in regressions:
    ratio = ca_m / oc_m if oc_m > 0 else float("inf")
    print(f"{cap:<20} {ca_m:>10} msgs {oc_m:>10} msgs {ratio:>6.1f}x")

avg_ca = sum(r[1] for r in regressions) / len(regressions)
avg_oc = sum(r[2] for r in regressions) / len(regressions)
print(f"\n{'Average':<20} {avg_ca:>10.0f} msgs {avg_oc:>10.0f} msgs {avg_ca/avg_oc:>6.1f}x")

# ── Scatter: CORE-Agent vs OpenCode messages for Opus 4.6 ──
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Panel 1: Opus 4.6 paired scatter
ax = axes[0]
for cap, ca_m, oc_m, ca_p, oc_p in all_pairs:
    if ca_p and oc_p:
        color, label = "#4CAF50", "Both pass"
    elif ca_p and not oc_p:
        color, label = "#F44336", "Regressed"
    elif not ca_p and oc_p:
        color, label = "#2196F3", "Recovered"
    else:
        color, label = "#757575", "Both fail"
    ax.scatter(ca_m, oc_m, c=color, s=70, alpha=0.7, edgecolors="white", linewidth=0.5, zorder=3)

lim = max(ax.get_xlim()[1], ax.get_ylim()[1])
ax.plot([0, lim], [0, lim], "k--", alpha=0.3, linewidth=1, zorder=1)
ax.set_xlabel("CORE-Agent message count")
ax.set_ylabel("OpenCode message count")
ax.set_title("Opus 4.6: CORE-Agent vs OpenCode\n(red = regressed, all below y=x)")
ax.spines[["top", "right"]].set_visible(False)

# Panel 2: Compare regression counts across target scaffolds
ax = axes[1]
scaffolds = ["Claude Code", "OpenCode"]
reg_counts = [3, 7]
rec_counts = [0, 0]
x = range(len(scaffolds))
ax.bar(x, reg_counts, color="#F44336", label="Regressions", edgecolor="white")
for i, v in enumerate(reg_counts):
    ax.text(i, v + 0.2, str(v), ha="center", fontweight="bold", fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(scaffolds)
ax.set_ylabel("Capsule count")
ax.set_title("Opus 4.6 Regressions from CORE-Agent\n(OpenCode is worse than Claude Code)")
ax.set_ylim(0, 9)
ax.spines[["top", "right"]].set_visible(False)

fig.suptitle("Opus 4.6: OpenCode Truncates Deep Iteration", fontsize=14, fontweight="bold", y=1.02)
fig.tight_layout()
fig.savefig("figures/opus46_opencode_regression.pdf", bbox_inches="tight")
plt.show()

print("\nThesis: OpenCode constrains Opus 4.6 more severely than Claude Code.")
print("Every regression shows 2-5x fewer messages. Where CORE-Agent allows 70-218 msgs,")
print("OpenCode caps at 18-50. This truncation eliminates the deep iteration that makes")
print("Opus 4.6 the top performer in CORE-Agent.")


In [ ]:

print("=" * 70)
print("GPT-5.4: Codex CLI High vs Medium Reasoning Effort")
print("=" * 70)

cdx_high = df[(df["model"] == "GPT-5.4") & (df["harness"] == "Codex CLI") & 
              (df["reasoning_effort"] == "high")].set_index("capsule_id")
cdx_med = df[(df["model"] == "GPT-5.4") & (df["harness"] == "Codex CLI") & 
             (df["reasoning_effort"] == "medium")].set_index("capsule_id")
common = cdx_high.index.intersection(cdx_med.index)

print(f"\nHigh: {cdx_high['passed'].sum()}/{len(cdx_high)} ({cdx_high['passed'].mean():.1%})")
print(f"Medium: {cdx_med['passed'].sum()}/{len(cdx_med)} ({cdx_med['passed'].mean():.1%})")
print(f"Avg msgs: high={cdx_high['message_count'].mean():.1f}, medium={cdx_med['message_count'].mean():.1f}")

# Differences
for cap in sorted(common):
    hp = cdx_high.loc[cap, "passed"]
    mp = cdx_med.loc[cap, "passed"]
    if hp != mp:
        hm = int(cdx_high.loc[cap, "message_count"])
        mm = int(cdx_med.loc[cap, "message_count"])
        print(f"\n  {cap}: high={'PASS' if hp else 'FAIL'} ({hm} msgs), medium={'PASS' if mp else 'FAIL'} ({mm} msgs)")

# Both fail
bf = [cap for cap in common if not cdx_high.loc[cap, "passed"] and not cdx_med.loc[cap, "passed"]]
if bf:
    print(f"\nBoth fail: {bf}")

# ── Paired scatter ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
for cap in common:
    hp = cdx_high.loc[cap, "passed"]
    mp = cdx_med.loc[cap, "passed"]
    hm = cdx_high.loc[cap, "message_count"]
    mm = cdx_med.loc[cap, "message_count"]
    if hp and mp:
        color = "#4CAF50"
    elif hp and not mp:
        color = "#2196F3"
    elif not hp and mp:
        color = "#FF9800"
    else:
        color = "#F44336"
    ax.scatter(mm, hm, c=color, s=60, alpha=0.7, edgecolors="white", zorder=3)

lim = max(ax.get_xlim()[1], ax.get_ylim()[1])
ax.plot([0, lim], [0, lim], "k--", alpha=0.3, linewidth=1, zorder=1)
ax.set_xlabel("Codex CLI (medium) message count")
ax.set_ylabel("Codex CLI (high) message count")
ax.set_title("GPT-5.4: Codex CLI High vs Medium\n(green=both pass, blue=high-only, red=both fail)")
ax.spines[["top", "right"]].set_visible(False)

# Panel 2: message count ratio histogram
ax = axes[1]
ratios = []
for cap in common:
    mm = cdx_med.loc[cap, "message_count"]
    hm = cdx_high.loc[cap, "message_count"]
    if mm > 0:
        ratios.append(hm / mm)
ax.hist(ratios, bins=15, color=MODEL_COLORS["GPT-5.4"], edgecolor="white", alpha=0.8)
ax.axvline(1.0, color="black", linestyle="--", alpha=0.4, linewidth=1)
if ratios:
    median_r = sorted(ratios)[len(ratios)//2]
    ax.axvline(median_r, color="red", linestyle="-", alpha=0.6, linewidth=1.5,
               label=f"Median: {median_r:.2f}x")
ax.set_xlabel("Message ratio (high / medium)")
ax.set_ylabel("Count")
ax.set_title("High uses ~14% more messages")
ax.legend(fontsize=9)
ax.spines[["top", "right"]].set_visible(False)

fig.suptitle("Codex CLI: Marginal Returns from Reasoning Effort",
             fontsize=14, fontweight="bold", y=1.02)
fig.tight_layout()
fig.savefig("figures/codex_high_vs_medium.pdf", bbox_inches="tight")
plt.show()

print("\nThesis: GPT-5.4 is near ceiling in Codex CLI. High reasoning effort adds")
print("~14% more messages and recovers exactly 1 capsule (capsule-9477017).")
print("Both effort levels fail only on capsule-4252248 (universally hardest).")


In [ ]:

print("=" * 80)
print("GRAND SUMMARY: All 10 Configurations Ranked by Pass Rate")
print("=" * 80)

summary = df.groupby("config").agg(
    pass_rate=("passed", "mean"),
    n_pass=("passed", "sum"),
    n_total=("passed", "count"),
    avg_msgs=("message_count", "mean"),
).sort_values("pass_rate", ascending=False)

# Add pass/fail message averages
for config in summary.index:
    sub = df[df["config"] == config]
    pm = sub[sub["passed"]]["message_count"].mean()
    fm = sub[~sub["passed"]]["message_count"].mean() if (~sub["passed"]).any() else float("nan")
    summary.loc[config, "avg_msgs_pass"] = pm
    summary.loc[config, "avg_msgs_fail"] = fm

print(f"\n{'Config':<35} {'Rate':>6} {'P/T':>6} {'Avg':>6} {'Pass':>6} {'Fail':>6}")
print("─" * 72)
for config, row in summary.iterrows():
    pr = f"{row['pass_rate']:.1%}"
    pt = f"{int(row['n_pass'])}/{int(row['n_total'])}"
    am = f"{row['avg_msgs']:.0f}"
    pm = f"{row['avg_msgs_pass']:.0f}"
    fm = f"{row['avg_msgs_fail']:.0f}" if not pd.isna(row['avg_msgs_fail']) else "—"
    print(f"  {config:<33} {pr:>6} {pt:>6} {am:>6} {pm:>6} {fm:>6}")

# ── Visualization: ranked bar chart ──
fig, ax = plt.subplots(figsize=(12, 7))
configs = list(summary.index)
rates = list(summary["pass_rate"])

# Color by model
def config_color(c):
    harness = c.split(" / ")[1].split(" (")[0]
    return SCAFFOLD_COLORS.get(harness, "#666666")

colors = [config_color(c) for c in configs]
bars = ax.barh(range(len(configs)), rates, color=colors, edgecolor="white", linewidth=1.5)
for i, (bar, rate) in enumerate(zip(bars, rates)):
    ax.text(rate + 0.005, i, f"{rate:.1%}", va="center", fontweight="bold", fontsize=10)
ax.set_yticks(range(len(configs)))
ax.set_yticklabels(configs, fontsize=10)
ax.set_xlabel("Pass rate", fontsize=12)
ax.set_xlim(0, 1.1)
ax.invert_yaxis()
ax.spines[["top", "right"]].set_visible(False)
ax.set_title("CoreBench Hard: All 10 Configurations Ranked", fontsize=14, fontweight="bold")

# Add legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=SCAFFOLD_COLORS[h], label=h) for h in SCAFFOLD_COLORS]
ax.legend(handles=legend_elements, loc="lower right", fontsize=10)

fig.tight_layout()
fig.savefig("figures/grand_summary_ranked.pdf", bbox_inches="tight")
plt.show()

# ── Final thesis ──
print("\n" + "=" * 80)
print("Final Thesis: Scaffold-Model Fit")
print("=" * 80)
print("""
1. SCAFFOLD CHOICE MATTERS AS MUCH AS MODEL CHOICE
   - GPT-5.4 spans 51.3% to 97.4% across scaffolds (46.1pp range)
   - Opus 4.6 spans 79.5% to 97.4% (17.9pp range)
   - Opus 4.5 spans 82.1% to 89.7% (7.6pp range)

2. THE RELATIONSHIP IS NOT MONOTONIC
   - GPT-5.4: first-party (Codex CLI) strictly better than CORE-Agent
   - Opus 4.6: third-party (CORE-Agent) strictly better than first-party (Claude Code)
   - OpenCode compresses all models to ~80-85% regardless of model

3. THREE MECHANISMS DRIVE SCAFFOLD EFFECTS
   a. Iteration depth: CORE-Agent allows 90+ msgs for Opus but GPT-5.4 only uses 36
   b. Tool availability: vision tool in CORE-Agent vs none in OpenCode/Codex CLI
   c. Infrastructure: timeout limits (2700s vs 18000s) and path lengths

4. THE "BEST" SCAFFOLD DEPENDS ON THE MODEL
   - For GPT-5.4: Codex CLI (high) at 97.4%
   - For Opus 4.6: CORE-Agent at 97.4%
   - For Opus 4.5: Claude Code at 89.7% (by a narrow margin)
   - No single scaffold is universally optimal
""")


---
# Part 4: Vision vs Written Question Analysis

22 of 39 capsules contain only vision questions (answer requires interpreting a figure), 13 contain only written questions, and 4 are mixed. Since OpenCode lacks a vision tool while CORE-Agent provides `query_vision_language_model`, this split is a natural experiment for measuring how much scaffold tool availability drives accuracy.

In [ ]:
# 2/10 pass rate. Only GPT-5.4/OpenCode and Opus 4.5/OpenCode pass.
# Three distinct failure modes: rounding, wrong layer, wrong curve.


# ── Step 1: collect all answers ─────────────────────────────────────────────────
cap_runs = [r for r in rows if r["agent_run_metadata"]["capsule_id"] == "capsule-4252248"]

CORRECT = 0.4929240513102846

answer_data = []
for r in cap_runs:
    meta = r["agent_run_metadata"]
    model_raw = meta.get("model", "")
    model = MODEL_MAP.get(model_raw, model_raw)
    harness = meta.get("harness", "")
    effort = meta.get("reasoning_effort") or "default"
    passed = meta.get("scores", {}).get("accuracy", 0.0) >= 1.0
    fa = meta.get("final_answer", {})
    val = float(list(fa.values())[0]) if isinstance(fa, dict) and fa else float("nan")
    msgs = meta.get("message_count", 0)

    if harness == "Codex CLI":
        config = f"{model} / {harness} ({effort})"
    else:
        config = f"{model} / {harness}"

    # Classify failure mode
    delta = abs(val - CORRECT)
    if passed:
        mode = "correct"
    elif abs(val - 0.493) < 0.001:
        mode = "rounding"
    elif abs(val - 0.4054) < 0.01:
        mode = "wrong layer"
    elif abs(val - 0.558) < 0.01:
        mode = "wrong curve"
    else:
        mode = "other"

    answer_data.append({
        "config": config, "model": model, "harness": harness,
        "answer": val, "correct": CORRECT, "delta": val - CORRECT,
        "rel_error": abs(val - CORRECT) / CORRECT * 100,
        "passed": passed, "mode": mode, "msgs": msgs
    })

adf = pd.DataFrame(answer_data)

# ── Step 2: print answer table ──────────────────────────────────────────────────
print("═══ CAPSULE-4252248: THE HARDEST CAPSULE (2/10 PASS) ═══")
print(f"Correct answer: {CORRECT}")
print(f"\n{'Config':<40} {'Answer':>14} {'Delta':>11} {'Rel Err':>8} {'Mode':<13} {'Pass'}")
print("-" * 100)
for _, row in adf.sort_values("delta").iterrows():
    print(f"  {row['config']:<38} {row['answer']:>12.10f} {row['delta']:>+11.7f} {row['rel_error']:>6.2f}%  {row['mode']:<13} {'PASS' if row['passed'] else 'FAIL'}")

# ── Step 3: failure mode summary ────────────────────────────────────────────────
print("\n═══ FAILURE MODE CLUSTERS ═══")
modes = adf[~adf["passed"]].groupby("mode").agg(
    count=("config", "count"),
    avg_answer=("answer", "mean"),
    configs=("config", lambda x: ", ".join(x))
).reset_index()
for _, m in modes.iterrows():
    print(f"\n  [{m['mode'].upper()}] ({m['count']} configs, avg answer: {m['avg_answer']:.4f})")
    print(f"    {m['configs']}")

# ── Step 4: Opus 4.6 rounding tragedy narrative ────────────────────────────────
print("\n═══ THE ROUNDING TRAGEDY: OPUS 4.6 / CORE-AGENT ═══")
for r in cap_runs:
    meta = r["agent_run_metadata"]
    if meta.get("model") != "claude-opus-4-6" or meta.get("harness") != "CORE-Agent":
        continue
    msgs_list = r.get("messages", [])

    # Find: (a) vision model reading, (b) R output with exact value, (c) final submission
    vision_reading = None
    exact_value = None
    final_sub = None

    for i, m in enumerate(msgs_list):
        content = m.get("content", "")
        if isinstance(content, list):
            content = " ".join(c.get("text", "") for c in content if isinstance(c, dict))
        if "Integration" in content and "0.493" in content and m.get("role") == "user" and vision_reading is None:
            vision_reading = (i, content[:300])
        if "0.4929241" in content and m.get("role") == "user" and exact_value is None:
            exact_value = (i, content[:300])
        if "0.493" in content and "final_answer" in content and m.get("role") == "assistant":
            final_sub = (i, content[:400])

    if vision_reading:
        print(f"  1. Vision model reads plot (msg {vision_reading[0]}):")
        for line in vision_reading[1].split("\n"):
            if "Integration" in line or "0.493" in line:
                print(f"     {line.strip()}")

    if exact_value:
        print(f"  2. R script computes exact value (msg {exact_value[0]}):")
        for line in exact_value[1].split("\n"):
            if "0.4929241" in line:
                print(f"     {line.strip()}")

    if final_sub:
        print(f"  3. Model submits ROUNDED value (msg {final_sub[0]}):")
        for line in final_sub[1].split("\n"):
            if "0.493" in line and ("answer" in line.lower() or "exact" in line.lower()):
                print(f"     {line.strip()}")

    print(f"\n  The model HAD 0.4929241 but CHOSE 0.493 from the vision tool reading.")
    print(f"  Scoring tolerance rejects 0.493 (0.02% relative error).")

# ── Step 5: figure ──────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# A: Answer values vs correct, colored by failure mode
ax = axes[0]
mode_colors = {"correct": "#4CAF50", "rounding": "#FF9800", "wrong layer": "#E53935", "wrong curve": "#9C27B0"}
mode_labels_seen = set()
for _, row in adf.sort_values("answer").iterrows():
    color = mode_colors.get(row["mode"], "#999")
    label = row["mode"] if row["mode"] not in mode_labels_seen else None
    mode_labels_seen.add(row["mode"])
    short = row["config"].replace("Opus 4.5", "O45").replace("Opus 4.6", "O46").replace("GPT-5.4", "G54").replace("CORE-Agent", "CA").replace("Claude Code", "CC").replace("Codex CLI", "CX").replace("OpenCode", "OC").replace(" (medium)", "m").replace(" (high)", "h").replace(" (max 10,000)", "").replace(" (adaptive)", "").replace(" / ", "/")
    ax.barh(short, row["answer"], color=color, alpha=0.85, edgecolor="black", linewidth=0.5, label=label)

ax.axvline(x=CORRECT, color="black", linestyle="--", linewidth=2, label=f"correct ({CORRECT:.4f})")
ax.set_xlabel("Submitted answer")
ax.set_title("A. Capsule-4252248: Submitted Answers by Failure Mode")
ax.legend(loc="lower right", fontsize=8)

# B: Relative error by config
ax = axes[1]
configs_sorted = adf.sort_values("rel_error")
colors_bar = [mode_colors.get(m, "#999") for m in configs_sorted["mode"]]
short_labels = [c.replace("Opus 4.5", "O45").replace("Opus 4.6", "O46").replace("GPT-5.4", "G54").replace("CORE-Agent", "CA").replace("Claude Code", "CC").replace("Codex CLI", "CX").replace("OpenCode", "OC").replace(" (medium)", "m").replace(" (high)", "h").replace(" (max 10,000)", "").replace(" (adaptive)", "").replace(" / ", "/") for c in configs_sorted["config"]]
ax.barh(short_labels, configs_sorted["rel_error"], color=colors_bar, alpha=0.85, edgecolor="black", linewidth=0.5)
ax.axvline(x=0.02, color="red", linestyle=":", linewidth=1.5, label="scoring threshold (~0.02%)")
ax.set_xlabel("Relative error (%)")
ax.set_title("B. Relative Err"
"or from Correct Answer")
ax.legend(fontsize=8)

for i, (_, row) in enumerate(configs_sorted.iterrows()):
    status = "PASS" if row["passed"] else "FAIL"
    ax.text(row["rel_error"] + 0.3, i, status, va="center", fontsize=8,
            color="#4CAF50" if row["passed"] else "#E53935", fontweight="bold")

plt.tight_layout()
plt.savefig("figures/capsule_4252248_hardest.pdf", bbox_inches="tight")
plt.show()

# ── Key findings ────────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("KEY FINDINGS")
print("=" * 60)
print("1. Three failure modes: rounding (Opus 4.6), wrong layer")
print("   (GPT-5.4), wrong curve (Opus 4.5 + GPT-5.4/CX med)")
print("2. Opus 4.6 computes exact answer (0.4929241) via R, then")
print("   discards it in favor of vision model reading (0.493).")
print("   The vision tool actively hurts on this capsule.")
print("3. Only OpenCode passes (2/2 models) — no vision tool")
print("   forces full-precision extraction from code output.")
print("4. Scoring tolerance is <0.02% relative error. 0.493 is")
print("   close but not close enough.")


In [ ]:
# i dont't think it actually resets the context - has to be checked in the hal repo
reset_analysis = []

core_df = df[df["harness"] == "CORE-Agent"]
core_runs_only = [r for r in rows if r["agent_run_metadata"].get("harness", "") == "CORE-Agent"]

for r in core_runs_only:
    meta = r["agent_run_metadata"]
    model = MODEL_MAP.get(meta["model"], meta["model"])
    cap = meta["capsule_id"]
    passed = meta["scores"]["accuracy"] >= 1.0
    msgs_list = r.get("messages", [])

    resets = 0
    pre_commands = set()
    segment = "pre"
    total_post = 0
    dupes = 0

    for m in msgs_list:
        if m.get("role") != "assistant":
            continue
        content = m.get("content", "")
        if isinstance(content, list):
            content = " ".join(c.get("text", "") for c in content if isinstance(c, dict))

        if "I still need to solve the task" in content:
            resets += 1
            segment = "post"
            continue

        if "execute_bash(" in content:
            for part in content.split("execute_bash(")[1:]:
                cmd = part[:80].strip().strip('"').strip("'")[:50]
                if segment == "pre":
                    pre_commands.add(cmd)
                else:
                    total_post += 1
                    if cmd in pre_commands:
                        dupes += 1
                    pre_commands.add(cmd)

    dupe_pct = dupes / total_post * 100 if total_post > 0 else 0
    productive_pct = 100 - dupe_pct
    reset_analysis.append({
        "model": model, "capsule": cap, "passed": passed,
        "msg_count": len(msgs_list), "resets": resets,
        "total_post_cmds": total_post, "duplicate_cmds": dupes,
        "dupe_pct": dupe_pct, "productive_pct": productive_pct
    })

ra_df = pd.DataFrame(reset_analysis)

print("=" * 70)
print("CONTEXT RESETS IN CORE-AGENT")
print("=" * 70)

print(f"\n{'Model':<12} {'Runs w/':>8} {'Avg':>6} {'Max':>5} {'Avg Dup%':>9} {'Productive%':>12}")
print(f"{'':12} {'resets':>8} {'reset':>6} {'':>5} {'post-reset':>9} {'commands':>12}")
print("-" * 60)

for model in ["Opus 4.5", "Opus 4.6", "GPT-5.4"]:
    mdf = ra_df[ra_df["model"] == model]
    wr = (mdf["resets"] > 0).sum()
    avg_r = mdf["resets"].mean()
    max_r = mdf["resets"].max()
    # Only runs with post-reset commands
    has_post = mdf[mdf["total_post_cmds"] > 0]
    avg_dup = has_post["dupe_pct"].mean() if len(has_post) > 0 else 0
    avg_prod = has_post["productive_pct"].mean() if len(has_post) > 0 else 0
    print(f"  {model:<10} {wr:>5}/{len(mdf):<3} {avg_r:>5.1f} {max_r:>5} {avg_dup:>8.0f}% {avg_prod:>10.0f}%")

print(f"\n{'=' * 70}")
print("THE EFFICIENCY GAP")
print(f"{'=' * 70}")

for model in ["Opus 4.6", "GPT-5.4"]:
    mdf = ra_df[ra_df["model"] == model]
    total_post = mdf["total_post_cmds"].sum()
    total_dupes = mdf["duplicate_cmds"].sum()
    net_new = total_post - total_dupes
    avg_msgs = mdf["msg_count"].mean()
    print(f"\n  {model}:")
    print(f"    Avg messages/run:        {avg_msgs:.0f}")
    print(f"    Avg resets/run:          {mdf['resets'].mean():.1f} (every ~11 msgs)")
    print(f"    Post-reset commands:     {total_post}")
    print(f"    Duplicate commands:      {total_dupes} ({total_dupes/total_post*100:.0f}%)")
    print(f"    Net new commands:        {net_new} ({net_new/total_post*100:.0f}% productive)")
    print(f"    New commands/run:        {net_new/len(mdf):.1f}")

print("""
Opus 4.6 makes 2.75x more effective progress per run than GPT-5.4.
GPT-5.4 "remembers" prior work (references it 100% of the time) but still
re-executes 67% of the same commands. The context reset destroys its working
memory of WHAT it ran, even though it retains a high-level sense of WHERE
it is in the task.
""")

# ── Step 4: figures ────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))

# A: Reset count distribution by model
ax = axes[0]
for model in ["Opus 4.5", "Opus 4.6", "GPT-5.4"]:
    mdf = ra_df[ra_df["model"] == model]
    ax.hist(mdf["resets"], bins=range(0, 23), alpha=0.5,
            color=MODEL_COLORS[model], label=model, edgecolor="black", linewidth=0.5)
ax.set_xlabel("Context resets per run")
ax.set_ylabel("Number of runs")
ax.set_title("A. Context Reset Distribution")
ax.legend()

ax = axes[1]
has_post = ra_df[ra_df["total_post_cmds"] > 0]
positions = {"Opus 4.5": 0, "Opus 4.6": 1, "GPT-5.4": 2}
for model in ["Opus 4.5", "Opus 4.6", "GPT-5.4"]:
    mdf = has_post[has_post["model"] == model]
    bp = ax.boxplot(mdf["dupe_pct"].values, positions=[positions[model]],
                    widths=0.6, patch_artist=True,
                    boxprops=dict(facecolor=MODEL_COLORS[model], alpha=0.6))
ax.set_xticks([0, 1, 2])
ax.set_xticklabels(["Opus 4.5", "Opus 4.6", "GPT-5.4"])
ax.set_ylabel("Command repetition rate (%)")
ax.set_title("B. Post-Reset Command Duplication")
ax.axhline(y=50, color="red", linestyle=":", alpha=0.5, label="50% threshold")
ax.legend()

ax = axes[2]
for model in ["Opus 4.5", "Opus 4.6", "GPT-5.4"]:
    mdf = has_post[has_post["model"] == model]
    for _, row in mdf.iterrows():
        marker = "o" if row["passed"] else "x"
        size = 60 if row["passed"] else 40
        ax.scatter(row["total_post_cmds"] - row["duplicate_cmds"], row["resets"],
                   c=MODEL_COLORS[model], marker=marker, s=size, alpha=0.6,
                   edgecolors="black" if row["passed"] else "none", linewidths=0.5)

ax.set_xlabel("Net new commands (post-reset)")
ax.set_ylabel("Number of resets")
ax.set_title("C. New Commands vs Reset Count")
# Custom legend
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker="o", color="w", markerfacecolor=MODEL_COLORS[m],
           markersize=8, label=m) for m in ["Opus 4.5", "Opus 4.6", "GPT-5.4"]
] + [
    Line2D([0], [0], marker="o", color="w", markerfacecolor="gray", markersize=8, label="Pass"),
    Line2D([0], [0], marker="x", color="gray", markersize=8, label="Fail", linestyle="None"),
]
ax.legend(handles=legend_elements, fontsize=7, loc="upper left")

plt.tight_layout()
plt.savefig("figures/context_resets_deep.pdf", bbox_inches="tight")
plt.show()

print("=" * 60)
print("KEY FINDINGS")
print("=" * 60)
print("1. Every CORE-Agent run hits context resets (every ~11 msgs)")
print("2. Both models reference prior work (100%), never truly restart")
print("3. GPT-5.4 repeats 67% of commands after resets (only 33%")
print("   productive). Opus 4.6 repeats only 28% (72% productive).")
print("4. Combined with shorter conversations, GPT-5.4 makes ~15")
print("   new commands/run vs Opus 4.6's ~43. The effective progress")
print("   gap is 2.75x, explaining much of the pass rate difference.")


In [ ]:
# Behavioral profiles for every harness, using metrics comparable across all scaffolds.


# ─── Compute cross-harness metrics from raw transcripts ──────────────────────
radar_records = []
for r in rows:
    meta = r["agent_run_metadata"]
    model = MODEL_MAP.get(meta.get("model", ""), meta.get("model", ""))
    harness = meta.get("harness", "")
    effort = meta.get("reasoning_effort") or "default"
    if harness == "Codex CLI":
        config = f"{model} / {harness} ({effort})"
    else:
        config = f"{model} / {harness}"
    
    passed = meta["scores"]["accuracy"] >= 1.0
    msg_count = meta.get("message_count", 0)
    messages = r.get("messages", [])
    
    error_encounters = 0
    error_recoveries = 0
    prev_had_error = False
    
    for m in messages:
        content = m.get("content", "")
        if isinstance(content, list):
            content = " ".join(c.get("text", "") for c in content if isinstance(c, dict))
        if m.get("role") == "assistant":
            if prev_had_error:
                error_recoveries += 1
                prev_had_error = False
        elif m.get("role") in ("user", "tool"):
            if any(kw in content.lower() for kw in 
                   ["error", "traceback", "exception", "permission denied", "not found"]):
                error_encounters += 1
                prev_had_error = True
            else:
                prev_had_error = False
    
    radar_records.append({
        "config": config, "model": model, "harness": harness,
        "passed": passed, "msg_count": msg_count,
        "error_encounters": error_encounters, 
        "error_recoveries": error_recoveries,
    })

rdf = pd.DataFrame(radar_records)

configs_agg = {}
for cfg in CONFIG_ORDER:
    sub = rdf[rdf["config"] == cfg]
    model = sub["model"].iloc[0]
    harness = sub["harness"].iloc[0]
    pass_sub = sub[sub["passed"]]
    fail_sub = sub[~sub["passed"]]
    
    pass_rate = sub["passed"].mean()
    avg_msgs = sub["msg_count"].mean()
    msg_cv = sub["msg_count"].std() / avg_msgs if avg_msgs > 0 else 0
    avg_errors = sub["error_encounters"].mean()
    recovery_rate = sub["error_recoveries"].sum() / sub["error_encounters"].sum() if sub["error_encounters"].sum() > 0 else 0
    
    fail_avg = fail_sub["msg_count"].mean() if len(fail_sub) > 0 else avg_msgs
    pass_avg = pass_sub["msg_count"].mean() if len(pass_sub) > 0 else avg_msgs
    effort_ratio = fail_avg / pass_avg if pass_avg > 0 else 1.0
    
    configs_agg[cfg] = {
        "model": model, "harness": harness,
        "Pass Rate": pass_rate,
        "Iteration Depth": avg_msgs,
        "Effort Ratio": effort_ratio,
        "Error Encounters": avg_errors,
        "Recovery Rate": recovery_rate,
        "Run Consistency": 1 - min(msg_cv, 1.0),  # invert: high = consistent
    }

# ─── Define harness groups ────────────────────────────────────────────────────
harness_groups = {
    "CORE-Agent": [c for c in CONFIG_ORDER if "CORE-Agent" in c],
    "Claude Code": [c for c in CONFIG_ORDER if "Claude Code" in c],
    "OpenCode": [c for c in CONFIG_ORDER if "OpenCode" in c],
    "Codex CLI": [c for c in CONFIG_ORDER if "Codex CLI" in c],
}

radar_dims = ["Pass Rate", "Iteration Depth", "Effort Ratio", "Error Encounters", "Recovery Rate", "Run Consistency"]

# ─── Global normalization (across ALL 10 configs for fair comparison) ─────────
global_min = {d: min(configs_agg[c][d] for c in CONFIG_ORDER) for d in radar_dims}
global_max = {d: max(configs_agg[c][d] for c in CONFIG_ORDER) for d in radar_dims}

def normalize(val, dim):
    mn, mx = global_min[dim], global_max[dim]
    if mx - mn < 1e-9:
        return 0.5
    return (val - mn) / (mx - mn)

fig, axes = plt.subplots(1, 4, figsize=(22, 5), subplot_kw=dict(polar=True))
fig.suptitle("Behavioral Profiles Across All Harnesses\n(globally normalized — shapes are comparable across panels)",
             fontsize=13, fontweight="bold", y=1.01)

angles = np.linspace(0, 2 * np.pi, len(radar_dims), endpoint=False).tolist()
angles += angles[:1]

for ax, (harness_name, configs) in zip(axes.flatten(), harness_groups.items()):
    for cfg in configs:
        model = configs_agg[cfg]["model"]
        values = [normalize(configs_agg[cfg][d], d) for d in radar_dims]
        values += values[:1]
        
        label = cfg.replace(" / ", "\n").replace("Codex CLI ", "CxCLI ")
        short = model
        if "Codex CLI" in cfg:
            short = "med" if "medium" in cfg else "high"
        
        ax.plot(angles, values, marker="o", linewidth=2, label=short, 
                color=MODEL_COLORS[model], 
                linestyle="--" if "medium" in cfg else "-",
                markersize=5)
        ax.fill(angles, values, alpha=0.08, color=MODEL_COLORS[model])
    
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(radar_dims, fontsize=8)
    ax.set_ylim(0, 1.15)
    ax.set_title(harness_name, fontsize=11, fontweight="bold", pad=15)
    ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.15), fontsize=8)

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig("figures/cross_harness_radars.pdf", bbox_inches="tight")
plt.show()
print("\n✓ Saved figures/cross_harness_radars.pdf")

# ─── Print normalized values for reference ────────────────────────────────────
print(f"\n{'Config':<32}", end="")
for d in radar_dims:
    print(f" {d[:12]:>12}", end="")
print()
print("-" * (32 + 13 * len(radar_dims)))
for cfg in CONFIG_ORDER:
    print(f"{cfg:<32}", end="")
    for d in radar_dims:
        print(f" {normalize(configs_agg[cfg][d], d):>12.2f}", end="")
    print()